## FPGA Matrix Multiplication Wrapper Function

In [1]:
from pynq import Overlay
from pynq import allocate
import numpy as np

In [2]:
def init_fpga(bitstream_path='/home/ubuntu/workspace/systolic_mmult.bit'):
    global overlay, dma, dma_send, dma_recv
    overlay = Overlay(bitstream_path)
    dma = overlay.axi_dma_0
    dma_send = dma.sendchannel
    dma_recv = dma.recvchannel

def pack_matrix(mat):
    """
    Packs a 4x4 matrix of INT8 values into a 1D array of 4 uint32 words.
    Each row (4 x 8-bit elements) is packed as: [elem3 << 24 | elem2 << 16 | elem1 << 8 | elem0]
    """
    # Cast to uint32 to ensure bitwise operations don't overflow signed representation
    mat_u32 = np.asarray(mat, dtype=np.uint8).astype(np.uint32)
    
    shifts = np.array([0, 8, 16, 24], dtype=np.uint32)
    # Shift columns and bitwise-OR along axis 1 (row-wise)
    return np.bitwise_or.reduce(mat_u32 << shifts, axis=1)

def unpack_matrix(buf):
    """
    Unpacks a 1D buffer of 8 uint64 words into a 4x4 matrix of INT32 accumulators.
    Each 64-bit word contains two 32-bit signed values:
      - Low 32 bits (bits 0..31)
      - High 32 bits (bits 32..63)
    8 words x 2 values = 16 elements (4x4 matrix).
    """
    buf64 = np.asarray(buf, dtype=np.uint64)
    
    # Extract lower 32 bits and upper 32 bits from each 64-bit word
    low_32  = (buf64 & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    high_32 = ((buf64 >> 32) & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    
    # Interleave low and high 32-bit words to reconstruct the 16 elements
    c_flat = np.empty(16, dtype=np.int32)
    c_flat[0::2] = low_32
    c_flat[1::2] = high_32
    
    return c_flat.reshape(4, 4)

def systolic_mmult(a, b):
    """
    Executes 4x4 matrix multiplication A @ B on hardware.
    - Inputs: a [4x4], b [4x4] (INT8)
    - Output: c [4x4] (INT32)
    """
    # 1. Allocate contiguous memory for DMA
    inp_buf = allocate(shape=(8,), dtype=np.uint32)  # 4 words B + 4 words A
    out_buf = allocate(shape=(8,), dtype=np.uint64)  # 8 words (16 x 32-bit outputs)
    
    # 2. Pack matrices: B first, then A (matches your working input_buffer)
    packed_b = pack_matrix(b)
    packed_a = pack_matrix(a)
    inp_buf[:4] = packed_b
    inp_buf[4:] = packed_a
    
    # 3. DMA Transfer
    dma_send.transfer(inp_buf)
    dma_recv.transfer(out_buf)
    dma_send.wait()
    dma_recv.wait()
    
    # 4. Unpack 64-bit output buffer into 4x4 INT32 matrix
    c = unpack_matrix(out_buf)
    
    # 5. Free DMA memory buffers
    del inp_buf, out_buf
    
    return c

In [3]:
init_fpga()

In [4]:
A = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]], dtype=np.int64)

B = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]], dtype=np.int64)

In [5]:
C = systolic_mmult(A, B)
C

array([[ 90, 100, 110, 120],
       [202, 228, 254, 280],
       [314, 356, 398, 440],
       [426, 484, 542, 600]], dtype=int32)

In [6]:
D = np.matmul(A, B)
D

array([[ 90, 100, 110, 120],
       [202, 228, 254, 280],
       [314, 356, 398, 440],
       [426, 484, 542, 600]])